# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsiddan25/program/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

Paste your Hugging Face READ token (hf_...): ··········


In [ ]:
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": (
        f"read_parquet('{REL}/dim_content.parquet')"
    ),
    "fact_daily": (
        f"read_parquet("
        f"'{REL}/fact_content_daily_performance/**/*.parquet'"
        f")"
    ),
}

print("DuckDB connection and table paths are ready.")

DuckDB connection and table paths are ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My feature window is Feb 1st - Apr 30th, 2026. The Outcome window is May 2026.
I want to prioritize pages that were previously visible in April (shown in google seach results and received impressions), not updated recently, and lost impressions from March to April ultimately had fewer impressions.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

page_windows = con.sql(f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-02-01'
                                     AND DATE '2026-02-28'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END) AS feb_days,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END) AS mar_days,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-04-01'
                                     AND DATE '2026-04-30'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END) AS apr_days,

            COUNT(DISTINCT CASE
                WHEN report_date BETWEEN DATE '2026-05-01'
                                     AND DATE '2026-05-31'
                 AND gsc_data_available IS TRUE
                THEN report_date
            END) AS may_days,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-02-01'
                                     AND DATE '2026-02-28'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions ELSE 0
            END) AS feb_impressions,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-03-01'
                                     AND DATE '2026-03-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions ELSE 0
            END) AS mar_impressions,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-04-01'
                                     AND DATE '2026-04-30'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions ELSE 0
            END) AS apr_impressions,

            SUM(CASE
                WHEN report_date BETWEEN DATE '2026-05-01'
                                     AND DATE '2026-05-31'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions ELSE 0
            END) AS may_impressions

        FROM {TABLES["fact_daily"]}
        WHERE report_date BETWEEN DATE '2026-02-01'
                              AND DATE '2026-05-31'
        GROUP BY 1, 2
    )

    SELECT
        m.*,
CASE
    WHEN d.content_updated_date <= DATE '2026-04-30'
    THEN DATE_DIFF(
        'day',
        d.content_updated_date,
        DATE '2026-04-30'
    )
    ELSE NULL
END AS days_since_last_update,

CASE
    WHEN d.content_updated_date > DATE '2026-04-30'
    THEN TRUE
    ELSE FALSE
END AS updated_after_feature_window,


        d.content_type,
        d.word_count,


    FROM monthly AS m
    INNER JOIN {TABLES["dim_content"]} AS d
        ON m.client_hash_id = d.client_hash_id
       AND m.content_hash_id = d.content_hash_id

    WHERE m.feb_days >= 20
      AND m.mar_days >= 20
      AND m.apr_days >= 20
      AND m.may_days >= 20
      AND m.mar_impressions > 0
      AND m.apr_impressions > 0

""").df()

print(f"Eligible active page histories: {len(page_windows):,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible active page histories: 62,558


In [ ]:
validation = {
    "rows": len(page_windows),
    "duplicate_pages": page_windows.duplicated(
        ["client_hash_id", "content_hash_id"]
    ).sum(),
    "negative_staleness_values": (
        page_windows["days_since_last_update"] < 0
    ).sum(),
    "missing_staleness_values": (
        page_windows["days_since_last_update"].isna()
    ).sum(),
}

validation

{'rows': 62558,
 'duplicate_pages': np.int64(0),
 'negative_staleness_values': np.int64(0),
 'missing_staleness_values': np.int64(46007)}

In [ ]:
page_windows["impressions_90d"] = (
    page_windows["feb_impressions"]
    + page_windows["mar_impressions"]
    + page_windows["apr_impressions"]
)

page_windows["recent_change_pct"] = (
    100
    * (
        page_windows["apr_impressions"]
        - page_windows["mar_impressions"]
    )
    / page_windows["mar_impressions"]
)

page_windows["future_change_pct"] = (
    100
    * (
        page_windows["may_impressions"]
        - page_windows["apr_impressions"]
    )
    / page_windows["apr_impressions"]
)

page_windows["future_decline"] = (
    page_windows["may_impressions"]
    < 0.80 * page_windows["apr_impressions"]
).astype(int)

print(
    "Future-decline base rate:",
    f"{100 * page_windows['future_decline'].mean():.2f}%"
)

staleness_data = page_windows.dropna(
    subset=["days_since_last_update"]
).copy()

staleness_data["staleness_bucket"] = pd.cut(
    staleness_data["days_since_last_update"],
    bins=[-np.inf, 90, 180, 365, np.inf],
    labels=[
        "less_than_90",
        "90_to_179",
        "180_to_364",
        "365_or_more",
    ],
    right=False,
)

staleness_table = (
    staleness_data
    .groupby("staleness_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        future_decline_rate=("future_decline", "mean"),
        median_future_change_pct=("future_change_pct", "median"),
    )
    .reset_index()
)

staleness_table["future_decline_rate"] = (
    100 * staleness_table["future_decline_rate"]
).round(2)

staleness_table["median_future_change_pct"] = (
    staleness_table["median_future_change_pct"].round(2)
)

print(f"Pages with known staleness: {len(staleness_data):,}")
staleness_table

Future-decline base rate: 48.79%
Pages with known staleness: 16,551


,staleness_bucket,n,future_decline_rate,median_future_change_pct
0,less_than_90,16543,49.11,-19.05
1,90_to_179,5,20.00,-4.69
2,180_to_364,3,66.67,-35.46


Signal 1 - Staleness: MIXED - This is a signal behind FlyRank's refresh flags, but it could not be dicerned in this chosen window. Of the 16, 551 pages with safe staleness values, only right were at least 90 days stale. The older buckets were too small to support a reliable conclusion, so staleness was removed from the baseline rule.

In [ ]:
page_windows["momentum_bucket"] = pd.cut(
    page_windows["recent_change_pct"],
    bins=[-np.inf, -50, -20, 0, 20, np.inf],
    labels=[
        "down_50_or_more",
        "down_20_to_50",
        "down_less_than_20",
        "up_0_to_20",
        "up_more_than_20",
    ],
    right=False,
)

momentum_table = (
    page_windows
    .groupby("momentum_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        future_decline_rate=("future_decline", "mean"),
        median_future_change_pct=("future_change_pct", "median"),
        median_apr_impressions=("apr_impressions", "median"),
    )
    .reset_index()
)

momentum_table["future_decline_rate"] = (
    100 * momentum_table["future_decline_rate"]
).round(2)

momentum_table["median_future_change_pct"] = (
    momentum_table["median_future_change_pct"].round(2)
)

print(f"Pages tested for momentum: {len(page_windows):,}")
momentum_table

Pages tested for momentum: 62,558


,momentum_bucket,n,future_decline_rate,median_future_change_pct,median_apr_impressions
0,down_50_or_more,13280,52.82,-23.53,313.0
1,down_20_to_50,19115,54.31,-24.39,832.0
2,down_less_than_20,10479,44.86,-15.30,1219.0
3,up_0_to_20,7150,39.71,-10.68,1384.5
4,up_more_than_20,12534,44.56,-13.57,1421.0


In [ ]:
threshold_results = []

for minimum_impressions in [1, 50, 100, 300, 500, 1000]:
    selected = page_windows[
        (page_windows["mar_impressions"] >= minimum_impressions)
        & (page_windows["recent_change_pct"] < -20)
    ]

    threshold_results.append({
        "minimum_march_impressions": minimum_impressions,
        "n_flagged": len(selected),
        "future_decline_rate": (
            100 * selected["future_decline"].mean()
            if len(selected) > 0 else np.nan
        ),
        "median_impression_loss": (
            selected["mar_impressions"]
            - selected["apr_impressions"]
        ).median() if len(selected) > 0 else np.nan,
    })

threshold_table = pd.DataFrame(threshold_results)

threshold_table["future_decline_rate"] = (
    threshold_table["future_decline_rate"].round(2)
)

threshold_table["median_impression_loss"] = (
    threshold_table["median_impression_loss"].round(1)
)

threshold_table

,minimum_march_impressions,n_flagged,future_decline_rate,median_impression_loss
0,1,32395,53.70,537.0
1,50,32391,53.71,537.0
2,100,31989,54.26,550.0
3,300,27356,58.67,707.0
4,500,23519,60.83,879.0
5,1000,17515,62.99,1291.0


Signal 2 - Recent Impression momentum: Confirmed. Pages whose impressions fell by at least 20% from March to April had May decline rates of 52.82%-54.31%. Pages that grew by 0%-20% had a lower decline rate of 39.71%. Recent negative momentum is associated with continued future decline, although it did not predict every page correctly.

Final Rule: Flag a page for refresh review if it received at least 300 impressions in March and lost more than 20% of its impressions from March to April. Rank flagged pages by their absolute impression loss.

Reason code: previously_visible_declining

Action label: review_for_refresh

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Final transparent rule thresholds
minimum_march_impressions = 300
decline_threshold_pct = -20

queue = page_windows.copy()

# A page qualifies when:
# 1. It had at least 300 March impressions.
# 2. Its impressions declined by more than 20% in April.
qualifies = (
    (queue["mar_impressions"] >= minimum_march_impressions)
    & (queue["recent_change_pct"] < decline_threshold_pct)
)

# Score = absolute number of impressions lost from March to April.
# Nonqualifying pages receive a score of zero.
queue["baseline_action_score"] = np.where(
    qualifies,
    queue["mar_impressions"] - queue["apr_impressions"],
    0
)

# One reason code for qualifying pages.
queue["reason_code"] = np.where(
    qualifies,
    "previously_visible_declining",
    ""
)

# The action is a human review, not an automatic refresh.
queue["action_label"] = np.where(
    qualifies,
    "review_for_refresh",
    "no_action"
)

# Rank highest-scoring pages first.
# April impressions break ties between equal scores.
queue = (
    queue
    .sort_values(
        ["baseline_action_score", "apr_impressions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

# Rank begins at 1 rather than Python's default 0.
queue.insert(
    0,
    "rank",
    np.arange(1, len(queue) + 1)
)

# Select only safe columns for the action queue.
# May outcome information is intentionally excluded.
queue_output = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_action_score",
        "reason_code",
        "action_label",
        "feb_impressions",
        "mar_impressions",
        "apr_impressions",
        "impressions_90d",
        "recent_change_pct",
        "days_since_last_update",
        "content_type",
        "word_count",
    ]
].copy()

# Confirm that no future-outcome fields entered the queue.
forbidden_columns = {
    "may_impressions",
    "future_change_pct",
    "future_decline",
}

assert forbidden_columns.isdisjoint(queue_output.columns)

# Create the required output folder and write the CSV.
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue_output.to_csv(output_path, index=False)

print(f"Total ranked pages: {len(queue_output):,}")
print(f"Pages flagged for review: {qualifies.sum():,}")
print(f"Highest score: {queue_output['baseline_action_score'].max():,.0f}")
print(f"Queue written to: {output_path}")
print("Leakage check passed: May outcome fields are excluded.")

Total ranked pages: 62,558
Pages flagged for review: 27,356
Highest score: 162,069
Queue written to: work/outputs/baseline_action_score.csv
Leakage check passed: May outcome fields are excluded.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# Select the first 20 pages recommended for review.
top20_review = (
    queue[
        queue["action_label"] == "review_for_refresh"
    ]
    .head(20)
    .copy()
)

# Explain quantitatively why each page qualified.
top20_review["why_it_is_here"] = top20_review.apply(
    lambda row: (
        f"March impressions fell from "
        f"{row['mar_impressions']:,.0f} to "
        f"{row['apr_impressions']:,.0f}; "
        f"a {abs(row['recent_change_pct']):.1f}% decline "
        f"and a loss of "
        f"{row['baseline_action_score']:,.0f} impressions."
    ),
    axis=1,
)

# These two columns require skeptical human review.
top20_review["confidence_note"] = "TO REVIEW"
top20_review["what_would_make_it_wrong"] = "TO REVIEW"

review_columns = [
    "rank",
    "content_hash_id",
    "action_label",
    "reason_code",
    "why_it_is_here",
    "content_type",
    "days_since_last_update",
    "word_count",
    "confidence_note",
    "what_would_make_it_wrong",
]

pd.set_option("display.max_colwidth", 200)

top20_review[review_columns]

,rank,content_hash_id,action_label,reason_code,why_it_is_here,content_type,days_since_last_update,word_count,confidence_note,what_would_make_it_wrong
0,1,content_44f34c0a90047651,review_for_refresh,previously_visible_declining,"March impressions fell from 212,404 to 50,335; a 76.3% decline and a loss of 162,069 impressions.",keyword article,<NA>,3495,TO REVIEW,TO REVIEW
1,2,content_ec2e0346994fb5a5,review_for_refresh,previously_visible_declining,"March impressions fell from 245,276 to 118,261; a 51.8% decline and a loss of 127,015 impressions.",keyword article,<NA>,2581,TO REVIEW,TO REVIEW
2,3,content_66288edeb93b7c4f,review_for_refresh,previously_visible_declining,"March impressions fell from 137,878 to 13,266; a 90.4% decline and a loss of 124,612 impressions.",keyword article,<NA>,2910,TO REVIEW,TO REVIEW
3,4,content_34a70fea29d15f24,review_for_refresh,previously_visible_declining,"March impressions fell from 143,019 to 28,278; a 80.2% decline and a loss of 114,741 impressions.",keyword article,<NA>,2639,TO REVIEW,TO REVIEW
4,5,content_f6116743b00afc2d,review_for_refresh,previously_visible_declining,"March impressions fell from 107,584 to 18,886; a 82.4% decline and a loss of 88,698 impressions.",keyword article,<NA>,2938,TO REVIEW,TO REVIEW
5,6,content_9c057b66c30a3abb,review_for_refresh,previously_visible_declining,"March impressions fell from 83,834 to 188; a 99.8% decline and a loss of 83,646 impressions.",keyword article,64,<NA>,TO REVIEW,TO REVIEW
6,7,content_7c6373141eae744a,review_for_refresh,previously_visible_declining,"March impressions fell from 132,593 to 49,347; a 62.8% decline and a loss of 83,246 impressions.",keyword article,<NA>,<NA>,TO REVIEW,TO REVIEW
7,8,content_cd3d932d4e1c8db0,review_for_refresh,previously_visible_declining,"March impressions fell from 89,332 to 13,447; a 84.9% decline and a loss of 75,885 impressions.",keyword article,<NA>,3222,TO REVIEW,TO REVIEW
8,9,content_e8a52cf3d5988c07,review_for_refresh,previously_visible_declining,"March impressions fell from 244,931 to 173,699; a 29.1% decline and a loss of 71,232 impressions.",keyword article,<NA>,3477,TO REVIEW,TO REVIEW
9,10,content_84a6bf3578312e90,review_for_refresh,previously_visible_declining,"March impressions fell from 91,388 to 22,117; a 75.8% decline and a loss of 69,271 impressions.",keyword article,<NA>,2776,TO REVIEW,TO REVIEW


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Re-create the top 20 internally for validation.
# May fields are allowed here because the pages have already been scored
# and ranked using only March-April information.
top20_evaluation = (
    queue[
        queue["action_label"] == "review_for_refresh"
    ]
    .head(20)
    .copy()
)

# A weak pick is a selected page that did not meet our
# future-decline definition in May.
weak_picks = top20_evaluation[
    top20_evaluation["future_decline"] == 0
].copy()

precision_at_20 = (
    top20_evaluation["future_decline"].mean()
)

print(f"Top-20 future declines: {top20_evaluation['future_decline'].sum()} of 20")
print(f"Precision@20: {precision_at_20:.2%}")
print(f"Weak picks: {len(weak_picks)}")

Top-20 future declines: 15 of 20
Precision@20: 75.00%
Weak picks: 5


In [ ]:
weak_pick_columns = [
    "rank",
    "content_hash_id",
    "baseline_action_score",
    "reason_code",
    "mar_impressions",
    "apr_impressions",
    "may_impressions",
    "recent_change_pct",
    "future_change_pct",
    "content_type",
    "days_since_last_update",
    "word_count",
]

weak_picks[weak_pick_columns]

,rank,content_hash_id,baseline_action_score,reason_code,mar_impressions,apr_impressions,may_impressions,recent_change_pct,future_change_pct,content_type,days_since_last_update,word_count
0,1,content_44f34c0a90047651,162069.0,previously_visible_declining,212404.0,50335.0,136735.0,-76.302235,171.649945,keyword article,<NA>,3495
2,3,content_66288edeb93b7c4f,124612.0,previously_visible_declining,137878.0,13266.0,14945.0,-90.378451,12.656415,keyword article,<NA>,2910
8,9,content_e8a52cf3d5988c07,71232.0,previously_visible_declining,244931.0,173699.0,202315.0,-29.082476,16.474476,keyword article,<NA>,3477
9,10,content_84a6bf3578312e90,69271.0,previously_visible_declining,91388.0,22117.0,39506.0,-75.798792,78.622779,keyword article,<NA>,2776
16,17,content_b99ea6861864dea5,52977.0,previously_visible_declining,194337.0,141360.0,118825.0,-27.260378,-15.941568,keyword article,<NA>,2680


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Fields that must not appear in the scored action queue.
forbidden_fields = {
    # Future outcome fields
    "may_impressions",
    "future_change_pct",
    "future_decline",

    # Label-derived or existing product-decision fields
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "health_score",
    "priority_score",
    "action_type",
}

leaked_fields = sorted(
    forbidden_fields.intersection(queue_output.columns)
)

print("Fields used to determine qualification:")
print([
    "mar_impressions",
    "apr_impressions",
    "recent_change_pct",
])

print("\nFields used to calculate the score:")
print([
    "mar_impressions",
    "apr_impressions",
])

print("\nForbidden fields found in exported queue:")
print(leaked_fields)

assert len(leaked_fields) == 0

print("\nLeakage check passed.")

Fields used to determine qualification:
['mar_impressions', 'apr_impressions', 'recent_change_pct']

Fields used to calculate the score:
['mar_impressions', 'apr_impressions']

Forbidden fields found in exported queue:
[]

Leakage check passed.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.